# Colab Overnight CheXNet Run

This notebook bootstraps a separate Colab search lane for `chestxray14`.
Use a unique session tag so it does not mix with a local overnight run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SESSION = 'chexnet_overnight_colab_20260310'
WORKDIR = '/content/autoresearch-medimage-colab'
DRIVE_ROOT = '/content/drive/MyDrive/autoresearch-medimage-colab'

In [ ]:
!rm -rf /content/autoresearch-medimage-colab
!git clone https://github.com/mlungrengmail/autoresearch-medimage.git /content/autoresearch-medimage-colab

In [ ]:
%cd /content/autoresearch-medimage-colab
!python -m pip install -q uv
!uv sync
!nvidia-smi

Upload `kaggle.json` to `MyDrive/kaggle.json` before running the next cell.

In [ ]:
!mkdir -p /root/.kaggle
!cp /content/drive/MyDrive/kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
%cd /content/autoresearch-medimage-colab
!uv run prepare.py --dataset chestxray14

In [ ]:
%cd /content/autoresearch-medimage-colab
!uv run train.py --recipe chexnet --session $SESSION --save-checkpoint > run.log 2>&1
!tail -n 120 run.log
!uv run python tools/mark_result.py --dataset chestxray14 --session $SESSION --status keep --append-note 'session baseline'

In [ ]:
!mkdir -p "$DRIVE_ROOT/results" "$DRIVE_ROOT/checkpoints" "$DRIVE_ROOT/artifacts"
!cp /content/autoresearch-medimage-colab/results.tsv "$DRIVE_ROOT/results/results.tsv"
!cp /content/autoresearch-medimage-colab/run.log "$DRIVE_ROOT/artifacts/run.log"
!cp -r /content/autoresearch-medimage-colab/checkpoints/. "$DRIVE_ROOT/checkpoints/" || true

Next step: start a fresh agent session against this Colab workspace and use the prompt in `docs/COLAB_OVERNIGHT_RUN.md`.